<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/wnut17-robust-ner/notebooks/wnut17_multi_model_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import numpy as np
import torch
from torch import nn
import transformers
from transformers import AutoTokenizer
import datasets

In [2]:
wnut = datasets.load_dataset("flaitenberger/wnut_17")

Repo card metadata block was not found. Setting CardData to empty.


In [3]:
label_list = [label for label in wnut["train"].features["ner_tags"].feature.names]
id2label =   {i:label for i, label in enumerate(label_list)}
label2id =   {label:i for i, label in id2label.items()}


In [4]:
shown = 0
for ex in wnut["train"]:

  if any(t != 0 for t in ex["ner_tags"]):
    print(" ".join(ex["tokens"]))
    print([(tok, id2label[t]) for tok, t in zip(ex["tokens"], ex["ner_tags"])])
    print()
  shown += 1

  if shown == 5:
    break

@paulwalk It 's the view from where I 'm living for two weeks . Empire State Building = ESB . Pretty bad storm here last evening .
[('@paulwalk', 'O'), ('It', 'O'), ("'s", 'O'), ('the', 'O'), ('view', 'O'), ('from', 'O'), ('where', 'O'), ('I', 'O'), ("'m", 'O'), ('living', 'O'), ('for', 'O'), ('two', 'O'), ('weeks', 'O'), ('.', 'O'), ('Empire', 'B-location'), ('State', 'I-location'), ('Building', 'I-location'), ('=', 'O'), ('ESB', 'B-location'), ('.', 'O'), ('Pretty', 'O'), ('bad', 'O'), ('storm', 'O'), ('here', 'O'), ('last', 'O'), ('evening', 'O'), ('.', 'O')]

From Green Newsfeed : AHFA extends deadline for Sage Award to Nov . 5 http://tinyurl.com/24agj38
[('From', 'O'), ('Green', 'O'), ('Newsfeed', 'O'), (':', 'O'), ('AHFA', 'B-group'), ('extends', 'O'), ('deadline', 'O'), ('for', 'O'), ('Sage', 'O'), ('Award', 'O'), ('to', 'O'), ('Nov', 'O'), ('.', 'O'), ('5', 'O'), ('http://tinyurl.com/24agj38', 'O')]

Pxleyes Top 50 Photography Contest Pictures of August 2010 ... http://bit.ly/b

In [5]:
for split in wnut:
  lens = [len(ex["tokens"]) for ex in wnut[split]]
  print(split, "mean:",round(np.mean(lens),2), "max: ", np.max(lens), "median: ", np.median(lens), "95th pct: ", np.quantile(lens, 0.95))

train mean: 18.48 max:  41 median:  18.0 95th pct:  30.0
validation mean: 15.59 max:  82 median:  13.0 95th pct:  38.0
test mean: 18.18 max:  105 median:  15.0 95th pct:  42.0


In [6]:
from collections import Counter

for split in wnut:
  c = Counter(id2label[t] for ex in wnut[split] for t in ex["ner_tags"])
  ents = {l[2:]: n for l , n in c.items() if l.startswith("B-")}
  total = sum(c.values())
  print(split, "tokens:", total, "O share:", round(c["O"] / total, 3))
  print(ents, "total entities:", sum(ents.values()))

train tokens: 62730 O share: 0.95
{'location': 548, 'group': 264, 'corporation': 221, 'person': 660, 'creative-work': 140, 'product': 142} total entities: 1975
validation tokens: 15733 O share: 0.921
{'location': 74, 'creative-work': 105, 'person': 470, 'product': 114, 'group': 39, 'corporation': 34} total entities: 836
test tokens: 23394 O share: 0.926
{'location': 150, 'group': 165, 'person': 429, 'creative-work': 142, 'corporation': 66, 'product': 127} total entities: 1079


In [7]:
tok = AutoTokenizer.from_pretrained("bert-base-cased")

words = ["Pxleyes", "Top", "50", "Photography", "Contest", "#photography"]

enc = tok(words, is_split_into_words= True)

print(tok.convert_ids_to_tokens(enc["input_ids"]))
print(enc.word_ids())

['[CLS]', 'P', '##xley', '##es', 'Top', '50', 'Photography', 'Contest', '#', 'photography', '[SEP]']
[None, 0, 0, 0, 1, 2, 3, 4, 5, 5, None]


In [8]:
def tokenize_and_align(batch):

  enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
  all_labels = []
  for i, tags in enumerate(batch["ner_tags"]):

    prev = None,
    labs = []

    for w in enc.word_ids(batch_index=i):

      if w is None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w

    all_labels.append(labs)
  enc["labels"] = all_labels

  return enc

In [9]:
tokenized = wnut.map(tokenize_and_align, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenized:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenized[split]["labels"]))
  print(split, "max_tokens:", max(len(x) for x in tokenized[split]["input_ids"]), "words lost:", lost)

Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

train max_tokens: 90 words lost: 0
validation max_tokens: 84 words lost: 0
test max_tokens: 198 words lost: 0


In [11]:
from seqeval.metrics import classification_report, accuracy_score
from seqeval.scheme import IOB2

def evaluate_ner(gold, pred):
    rep = classification_report(gold, pred, mode="strict", scheme=IOB2, output_dict=True, zero_division=0)
    out = {"precision": rep["micro avg"]["precision"], "recall": rep["micro avg"]["recall"], "f1": rep["micro avg"]["f1-score"], "token_acc": accuracy_score(gold, pred)}
    out["per_type_f1"] = {k: v["f1-score"] for k, v in rep.items() if "avg" not in k}
    return out

gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["validation"]]


{'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'token_acc': 1.0, 'per_type_f1': {'corporation': np.float64(1.0), 'creative-work': np.float64(1.0), 'group': np.float64(1.0), 'location': np.float64(1.0), 'person': np.float64(1.0), 'product': np.float64(1.0)}}
{'precision': np.float64(0.0), 'recall': np.float64(0.0), 'f1': np.float64(0.0), 'token_acc': 0.9205491641772071, 'per_type_f1': {'corporation': np.float64(0.0), 'creative-work': np.float64(0.0), 'group': np.float64(0.0), 'location': np.float64(0.0), 'person': np.float64(0.0), 'product': np.float64(0.0)}}
